# Fig. 8: halo MSLE against radial motion

(a) The per-halo MSLE of Form 1 (Table 3, read from `table3_fig6_generalised_forms`'s `output/forms_per_halo.csv`)
against $\langle v_r\rangle/\sigma_r$ of the bound set, in the proper frame $v_r + H r$ used by the bound
selection ($<0$ is net infall). (b) Halo 1's projected density. Also prints the correlations of the MSLE
with $\langle v_r\rangle/\sigma_r$, $\beta$ and $c/a$, and with assembly history.

Needs `$AURIGA_DATA`. Writes `gnuplot/{mse_scatter,mse_scatter_labels,halo01_projected_grid}.dat` and
`gnuplot/_meta.gnu` (read by `gnuplot/figure8_combined.gnu`), and `output/fig8_halos.csv`, `output/fig8_assembly.csv`.

In [ ]:
import os
import sys
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr

sys.path.insert(0, str(Path.cwd().resolve().parents[1]))  # AurigaDFs/, so the package imports
from Empirical_DF_Construction.paper_figures.common import SNAP, bound_particles, out_dir

GP = Path.cwd() / "gnuplot"
GP.mkdir(exist_ok=True)
MSLE = pd.read_csv(out_dir() / "forms_per_halo.csv", index_col="halo")["msle1"]   # Form 1, Table 3
print(f"mean MSLE (Form 1) = {MSLE.mean():.4f}")

## Kinematics of the bound set

In [ ]:
def kinematics(h):
    # <v_r>/sigma_r in the proper frame, beta, and c/a from the reduced inertia tensor
    d = bound_particles(h)
    x, v, m = d["x_kpc"], d["v_kms"], d["m_msun"]
    r = np.linalg.norm(x, axis=1)
    vr = np.einsum("ij,ij->i", x, v) / r
    H = 0.1 * float(d["h"]) * np.sqrt(float(d["Omega0"]) / float(d["a"]) ** 3 + float(d["OmegaL"]))  # km/s/kpc
    vr_p = vr + H * r
    vt2 = np.maximum(np.einsum("ij,ij->i", v, v) - vr ** 2, 0)
    w = m / r ** 2
    lam = np.linalg.eigvalsh(np.einsum("k,ki,kj->ij", w, x, x) / w.sum())
    return dict(proxy=vr_p.mean() / vr_p.std(), beta=1 - vt2.mean() / (2 * vr.var()),
                c_over_a=np.sqrt(lam[0] / lam[2]))


T = pd.DataFrame({h: kinematics(h) for h in MSLE.index}).T.assign(msle1=MSLE)
T.to_csv(out_dir() / "fig8_halos.csv", index_label="halo")
for c in ("proxy", "beta", "c_over_a"):
    rho, p = spearmanr(T[c], T.msle1)
    print(f"MSLE vs {c:8s}: Pearson r = {pearsonr(T[c], T.msle1)[0]:+.3f}, Spearman rho = {rho:+.3f} (p = {p:.3g})")

## Figure data

In [ ]:
# (a) scatter, with the five worst fits labelled
worst = T.msle1.nlargest(5).index
np.savetxt(GP / "mse_scatter.dat", np.column_stack([T.proxy, T.msle1, T.index]),
           fmt=["%.8g", "%.8g", "%d"], header="mean_vr_proper_over_sigma_r  halo_msle_form1  halo_id")
np.savetxt(GP / "mse_scatter_labels.dat", np.column_stack([T.proxy[worst], T.msle1[worst], worst]),
           fmt=["%.8g", "%.8g", "%d"], header="label_x label_y halo_id")

# (b) Halo 1's projected density, +-1000 kpc on a 512^2 grid
HALF, NB, VMIN, VMAX = 1000.0, 512, 5.0, 7.5
d1 = bound_particles(1)
x, m = d1["x_kpc"], d1["m_msun"]
k = np.all(np.abs(x) < HALF, axis=1)
edges = np.linspace(-HALF, HALF, NB + 1)
Sigma = np.histogram2d(x[k, 0], x[k, 1], bins=edges, weights=m[k])[0].T / (2 * HALF / NB) ** 2
logS = np.where(Sigma > 0, np.log10(np.maximum(Sigma, 1e-30)), VMIN)
mid = 0.5 * (edges[:-1] + edges[1:])
with (GP / "halo01_projected_grid.dat").open("w") as fp:
    fp.write("# x_kpc y_kpc log10_sigma\n")
    for j, yy in enumerate(mid):
        fp.writelines(f"{xx:.6g} {yy:.6g} {logS[j, i]:.6g}\n" for i, xx in enumerate(mid))
        fp.write("\n")
(GP / "_meta.gnu").write_text("\n".join([
    f"PROXY_MIN = {T.proxy.min():.8g}", f"PROXY_MAX = {T.proxy.max():.8g}",
    f"MSE_MIN = {T.msle1.min():.8g}", f"MSE_MAX = {T.msle1.max():.8g}",
    f"SIGMA_VMIN = {VMIN}", f"SIGMA_VMAX = {VMAX}", f"MAP_EXTENT = {HALF}", f"NBINS = {NB}"]) + "\n")

fig, (a, b) = plt.subplots(1, 2, figsize=(10, 4.5))
a.scatter(T.proxy, T.msle1, s=40, edgecolors="k")
for h in worst:
    a.annotate(str(h), (T.proxy[h], T.msle1[h]), xytext=(4, 4), textcoords="offset points")
a.set(xlabel=r"$\langle v_r\rangle/\sigma_r$", ylabel="Halo MSLE (Form 1)")
b.imshow(np.where(Sigma > 0, logS, np.nan), extent=[-HALF, HALF, -HALF, HALF], origin="lower",
         cmap="hot", vmin=VMIN, vmax=VMAX)
b.set(xlabel="x [kpc]", ylabel="y [kpc]")
plt.show()

## Assembly history

Main branch of the $z=0$ halo in the merger trees (`trees_sf1_127.0.hdf5`, LHaloTree: `Tree0` row 0, then
`FirstProgenitor`), keeping only snaps where it is its FoF group's central, so `Group_M_Crit200` is its own
mass. Per halo: $z_{50}$, where $M_{200c}$ last crossed $M_0/2$, and $f_{z<0.5} = 1 - M(z{=}0.5)/M_0$.
Trees are read from `$AURIGA_DATA/Halo N/`, or `$AURIGA_TREES` if kept elsewhere.

In [ ]:
def tree_path(h):
    for base in filter(None, (os.getenv("AURIGA_DATA"), os.getenv("AURIGA_TREES"))):
        for name in (f"Halo {h}", f"Halo_{h}", f"Halo_{h:02d}"):
            if (p := Path(base) / name / "trees_sf1_127.0.hdf5").exists():
                return p


def assembly(h):
    with h5py.File(tree_path(h), "r") as f:
        g, z_of_snap = f["Tree0"], f["Header/Redshifts"][:]
        assert g["SnapNum"][0] == SNAP and g["SubhaloNumber"][0] == 0, f"Halo {h}: Tree0[0] is not the z=0 halo"
        fp, br = g["FirstProgenitor"][:], [0]
        while fp[br[-1]] >= 0:
            br.append(fp[br[-1]])
        br = np.array(br[::-1])                                   # oldest first
        central = g["FirstHaloInFOFGroup"][:][br] == br
        z, lnm = z_of_snap[g["SnapNum"][:][br]][central], np.log(g["Group_M_Crit200"][:][br][central])
    half = lnm[-1] - np.log(2)
    i = np.where(lnm < half)[0][-1]                                # last snap below M0/2
    z50 = z[i] + (half - lnm[i]) / (lnm[i + 1] - lnm[i]) * (z[i + 1] - z[i])
    f05 = 1 - np.exp(np.interp(0.5, z[::-1], lnm[::-1]) - lnm[-1])
    return dict(z50=z50, frac_gained_since_z05=f05)


A = pd.DataFrame({h: assembly(h) for h in T.index if tree_path(h)}).T.join(T[["msle1", "proxy"]])
A.to_csv(out_dir() / "fig8_assembly.csv", index_label="halo")
for c in ("z50", "frac_gained_since_z05"):
    rho, p = spearmanr(A[c], A.msle1)
    print(f"MSLE vs {c}: Spearman rho = {rho:+.2f} (p = {p:.2g})")
rho, p = spearmanr(A.frac_gained_since_z05, A.proxy)
print(f"<v_r>/sigma_r vs f_(z<0.5): Spearman rho = {rho:+.2f} (p = {p:.2g})")